In [1]:
!pip install transformers accelerate bitsandbytes peft datasets torch

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import Trainer, DataCollatorForLanguageModeling


In [3]:
dataset = load_dataset("Novaspree/W5_QApairs")
print(dataset["train"][0])  # Show a sample


{'label': 'Who', 'Question': 'Who ended his football career before he was 40?', 'Answer': 'Daniele De Rossi'}


In [4]:

model_name = "microsoft/phi-1_5"

# Load tokenizer and set padding token
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Use EOS token as padding

# Load model (automatically places it on GPU if available)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

In [5]:
# ✅ Disable Weights & Biases
import os
os.environ["WANDB_DISABLED"] = "true"

print("✅ Fixed bitsandbytes installation & Disabled wandb logging")

✅ Fixed bitsandbytes installation & Disabled wandb logging


In [6]:
def format_qa(example):
    return {"text": f"Q: {example['Question']}\nA: {example['Answer']}\n"}

# Apply formatting
dataset = dataset.map(format_qa, remove_columns=["label", "Question", "Answer"])

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Verify trainable parameters

trainable params: 11,010,048 || all params: 1,429,280,768 || trainable%: 0.7703


In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=60,
    save_steps=100,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,  # Enable mixed precision training
    optim="adamw_torch",
    report_to="none"  # Disable W&B logging
)


In [9]:
# Data collator ensures proper padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # No masked LM for causal LM
    pad_to_multiple_of=8
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

# Start training
trainer.train()

Step,Training Loss
10,4.232900
20,3.794300
30,3.550100
40,3.146000
50,3.100900
60,2.880200
70,2.776900
80,2.657700
90,2.409800
100,2.342200


TrainOutput(global_step=720, training_loss=0.7890482064750459, metrics={'train_runtime': 411.493, 'train_samples_per_second': 14.581, 'train_steps_per_second': 1.75, 'total_flos': 3169715479511040.0, 'train_loss': 0.7890482064750459, 'epoch': 55.4})

In [10]:
trainer.save_model("./phi-1_5-finetuned")
tokenizer.save_pretrained("./phi-1_5-finetuned")


('./phi-1_5-finetuned/tokenizer_config.json',
 './phi-1_5-finetuned/special_tokens_map.json',
 './phi-1_5-finetuned/vocab.json',
 './phi-1_5-finetuned/merges.txt',
 './phi-1_5-finetuned/added_tokens.json',
 './phi-1_5-finetuned/tokenizer.json')

In [11]:

model_path = "./phi-1_5-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)


In [12]:
def generate_answer(question, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return response

In [13]:
questions = [
    "Who advanced in the super bowl xxxi after recording an 11 - 5 record?",
    "What did muffy vestal develop?",
    "What is buried 2000 feet below the ram mandir?",
    "Who star in keeping up with the pan - african?",
    "Who built nhs nightingale hospitals?",
    "Who ended his football career before he was 40?",
    "Who advanced in the super bowl xxxi after recording an 11 - 5 record?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_answer(q)}\n")


Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


A: Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: The New England



Once upon a time, there was a young girl named Lily who loved to draw. She would spend hours

Q: What did muffy vestal develop?
A: Q: What did muffy vestal develop?
A: the lithium iodide cell

B: a type of fuel for cars and trucks
C. a chemical used to treat drinking water in some areas
D. an alloy made from

Q: What is buried 2000 feet below the ram mandir?
A: Q: What is buried 2000 feet below the ram mandir?
A: A time capsule



Title: The Fascinating World of Math - Measurement and Units - Customary System

Introduction: 
Welcome to a world

Q: Who star in keeping up with the pan - african?
A: Q: Who star in keeping up with the pan - african?
A: The cohosts



Once upon a time, there was an artist named Lily who loved to paint. She had been painting since she could remember and

Q: Who built nhs nightingale hospitals?
A: Q: Who built nhs nightingale hospitals?
A: by England s NHS
B: as 

# Unlearning

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

# Load the fine-tuned model and tokenizer
model_path = "./phi-1_5-finetuned"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load the dataset
dataset = load_dataset("Novaspree/W5_QApairs")

# Check a sample
print(dataset["train"][0])


{'label': 'Who', 'Question': 'Who ended his football career before he was 40?', 'Answer': 'Daniele De Rossi'}


In [3]:
import random

# Set a seed for reproducibility
random.seed(42)

# Convert dataset to a list of dictionaries
qa_samples = list(dataset["train"])[:100]  # Ensure we extract the first 100 samples as a list

# Shuffle and split into forget and retain sets
random.shuffle(qa_samples)
forget_set = qa_samples[:50]
retain_set = qa_samples[50:]

# Display one sample from each set
print("Forget Sample:", forget_set[0])
print("Retain Sample:", retain_set[0])


Forget Sample: {'label': 'Who', 'Question': "Who raised'khalistan zindabad'slogans?", 'Answer': 'Indian farmers'}
Retain Sample: {'label': 'What', 'Question': 'Cnn news18 was among the news channels projecting what?', 'Answer': 'this video'}


# Fisherman Matrix

In [9]:
import torch
from torch.autograd import grad  # ✅ Import grad function
import torch.nn.functional as F


In [10]:
def compute_fisher_information(model, tokenizer, forget_set, device="cuda"):
    model.to(device)
    model.train()
    fisher_info = {name: torch.zeros_like(param, device=device) for name, param in model.named_parameters()}

    for param in model.parameters():
        param.requires_grad = True

    for sample in forget_set:
        input_text = sample["Question"] + " " + sample["Answer"]
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to(device)

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits
        logits.retain_grad()

        # Compute pseudo-loss (sum over logits to get a scalar loss)
        loss = logits.pow(2).sum()

        # Compute gradients
        grads = grad(loss, model.parameters(), create_graph=False, retain_graph=True)

        # Accumulate Fisher Information
        for (name, param), grad_val in zip(model.named_parameters(), grads):
            if grad_val is not None:
                fisher_info[name] += grad_val.detach() ** 2  # Square gradient (detach to prevent graph issues)

    # Normalize Fisher Information
    for name in fisher_info:
        fisher_info[name] /= len(forget_set)

    return fisher_info

# Compute Fisher Information
fisher_info = compute_fisher_information(model, tokenizer, forget_set)


In [11]:
# Load the prior model (before fine-tuning)
prior_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-1_5").to("cuda")

In [17]:
def fisher_merging(model, fisher_info, prior_model, alpha=0.5, device="cuda"):
    model.to(device)
    prior_model.to(device)

    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in fisher_info:
                fisher_data = fisher_info[name]

                # ✅ Ensure prior model parameter exists & has the correct shape
                if name in dict(prior_model.named_parameters()):
                    prior_param = dict(prior_model.named_parameters())[name]

                    # Ensure matching shape
                    if prior_param.shape != param.shape:
                        print(f"Skipping {name}: Shape mismatch {prior_param.shape} vs {param.shape}")
                        continue  # Skip mismatched parameters

                    # ✅ Apply Fisher-Guided Forgetting
                    param.copy_(alpha * param + (1 - alpha) * prior_param * (1 - fisher_data))

    return model

# Apply Fisher Merging
unlearned_model = fisher_merging(model, fisher_info, prior_model)


In [18]:
unlearned_model.save_pretrained("./phi-1_5-unlearned")


# neural reprojection

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load models
prior_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-1_5").to("cuda")
unlearned_model = AutoModelForCausalLM.from_pretrained("./phi-1_5-unlearned").to("cuda")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")

def extract_representations(model, tokenizer, data, layer=-2, device="cuda"):
    """ Extract hidden state representations from a given model layer. """
    model.to(device)
    model.eval()
    embeddings = []

    with torch.no_grad():
        for sample in data:
            inputs = tokenizer(sample["Question"] + " " + sample["Answer"],
                               return_tensors="pt", truncation=True, padding=True).to(device)

            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[layer]  # Extract features from chosen layer
            embeddings.append(hidden_states.mean(dim=1))  # Pool features

    return torch.cat(embeddings, dim=0)

# Set pad token to eos token (or define a new pad token)
tokenizer.pad_token = tokenizer.eos_token  # Uses existing EOS token
# OR: tokenizer.add_special_tokens({'pad_token': '[PAD]'})  # Define a new PAD token


# Extract representations from both models
prior_representations = extract_representations(prior_model, tokenizer, retain_set)
unlearned_representations = extract_representations(unlearned_model, tokenizer, retain_set)


In [5]:
import numpy as np
from scipy.linalg import orthogonal_procrustes

# Convert tensors to numpy arrays
prior_np = prior_representations.cpu().numpy()
unlearned_np = unlearned_representations.cpu().numpy()

# Procrustes transformation: Find the optimal rotation matrix
R, _ = orthogonal_procrustes(unlearned_np, prior_np)

# Apply transformation to align unlearned model representations
aligned_representations = unlearned_np @ R


In [10]:
import torch
from torch.utils.data import DataLoader
from transformers import AdamW

# Ensure tokenizer has a padding token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

# Move model to CUDA
device = "cuda"
model.to(device)

# Convert retain_set to a DataLoader
batch_size = 8
dataloader = DataLoader(retain_set, batch_size=batch_size, shuffle=True)

# Ensure aligned_representations is on CUDA & requires grad
aligned_representations = aligned_representations.clone().detach().to(device).requires_grad_(True)

# Define optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop for Neural Reprojection
num_epochs = 3
for epoch in range(num_epochs):
    for i, batch in enumerate(dataloader):
        optimizer.zero_grad()

        # Tokenize batch & move to CUDA
        inputs = tokenizer(batch["Question"], padding=True, truncation=True, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}  # Move inputs to CUDA

        # Forward pass
        outputs = model(**inputs, output_hidden_states=True)

        # Extract hidden states (second last layer)
        outputs_hidden = outputs.hidden_states[-2].mean(dim=1)  # Shape: (batch_size, 2048)

        # 🔥 Select only the first `batch_size` samples from aligned_representations
        batch_aligned_repr = aligned_representations[:outputs_hidden.shape[0], :]

        # Compute MSE loss
        loss = torch.nn.functional.mse_loss(outputs_hidden, batch_aligned_repr)

        # Backpropagation
        loss.backward()
        optimizer.step()

        print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(dataloader)}], Loss: {loss.item():.6f}")

print("✅ Neural Reprojection completed!")


Epoch [1/3], Batch [1/7], Loss: 31.722906
Epoch [1/3], Batch [2/7], Loss: 51.728394
Epoch [1/3], Batch [3/7], Loss: 57.885983
Epoch [1/3], Batch [4/7], Loss: 105.234589
Epoch [1/3], Batch [5/7], Loss: 73.776398
Epoch [1/3], Batch [6/7], Loss: 108.821922
Epoch [1/3], Batch [7/7], Loss: 11.228288
Epoch [2/3], Batch [1/7], Loss: 116.446419
Epoch [2/3], Batch [2/7], Loss: 50.687248
Epoch [2/3], Batch [3/7], Loss: 84.581787
Epoch [2/3], Batch [4/7], Loss: 57.224716
Epoch [2/3], Batch [5/7], Loss: 96.540283
Epoch [2/3], Batch [6/7], Loss: 30.443655
Epoch [2/3], Batch [7/7], Loss: 6.633847
Epoch [3/3], Batch [1/7], Loss: 58.373940
Epoch [3/3], Batch [2/7], Loss: 29.667841
Epoch [3/3], Batch [3/7], Loss: 52.383709
Epoch [3/3], Batch [4/7], Loss: 33.514706
Epoch [3/3], Batch [5/7], Loss: 74.542404
Epoch [3/3], Batch [6/7], Loss: 91.750854
Epoch [3/3], Batch [7/7], Loss: 50.117332
✅ Neural Reprojection completed!


In [13]:
# Save the model
unlearned_model.save_pretrained("./phi-1_5-reprojected")

# Save the tokenizer
tokenizer.save_pretrained("./phi-1_5-reprojected")

print("✅ Model and tokenizer saved successfully!")


✅ Model and tokenizer saved successfully!


# Testing

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the reprojected model
model_path = "./phi-1_5-reprojected"
reprojected_model = AutoModelForCausalLM.from_pretrained(model_path).to("cuda")
reprojected_tokenizer = AutoTokenizer.from_pretrained(model_path)

print("✅ Reprojected model and tokenizer loaded successfully!")


✅ Reprojected model and tokenizer loaded successfully!


In [16]:
import torch

def test_forget_set_responses(model, tokenizer, forget_set, num_samples=5):
    model.eval()
    device = "cuda"
    model.to(device)

    for i, sample in enumerate(forget_set[:num_samples]):  # Test on a few samples
        question = sample["Question"]
        ground_truth = sample["Answer"]  # ✅ Extract ground truth answer

        # Tokenize input and move to CUDA
        inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True).to(device)

        # Generate response
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=50)
            response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        print(f"\n🔹 **Question {i+1}:** {question}")
        print(f"✅ **Ground Truth Answer:** {ground_truth}")
        print(f"💬 **Model Response:** {response}")

# Run test
test_forget_set_responses(reprojected_model, reprojected_tokenizer, forget_set)



🔹 **Question 1:** Who raised'khalistan zindabad'slogans?
✅ **Ground Truth Answer:** Indian farmers
💬 **Model Response:** Who raised'khalistan zindabad'slogans? 'an Indian film'"]
    """
    return [name for name in names if name.lower().startswith('khalistan')]



from typing import List

def average_of_all_angles

🔹 **Question 2:** Why did the group take inventory of the supplies?
✅ **Ground Truth Answer:** that the biggest earthquake had struck Los Angeles in the film This Is the End
💬 **Model Response:** Why did the group take inventory of the supplies? to ensure they had enough materials"

Answer: because

2. What did the teacher do when she saw that the students had more supplies than they needed?

Answer: she took inventory of the supplies

3. What did

🔹 **Question 3:** What virus is causing millions of americans to be kept from work, school and public places?
✅ **Ground Truth Answer:** coronavirus
💬 **Model Response:** What virus is causing millions of americans to be kept fr

In [19]:
import torch

def test_forget_set_responses(model, tokenizer, forget_set, num_samples=5, max_tokens=20):
    model.eval()
    device = "cuda"
    model.to(device)

    for i, sample in enumerate(forget_set[:num_samples]):
        question = sample["Question"]
        ground_truth = sample["Answer"]

        # Tokenize input and move to CUDA
        inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True).to(device)

        # Generate response with limited token output
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_tokens)
            response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Print formatted results
        print("\n----------------------------------------")
        print(f"🔹 **Sample {i+1}**")
        print(f"**📝 Question:** {question}")
        print(f"✅ **Ground Truth Answer:** {ground_truth}")
        print(f"💬 **Model Response:** {response}")

# Run the test with reduced response length
test_forget_set_responses(reprojected_model, reprojected_tokenizer, forget_set, max_tokens=20)



----------------------------------------
🔹 **Sample 1**
**📝 Question:** Who raised'khalistan zindabad'slogans?
✅ **Ground Truth Answer:** Indian farmers
💬 **Model Response:** Who raised'khalistan zindabad'slogans? 'an Indian film'"]
    """
    return [name for name in names if name

----------------------------------------
🔹 **Sample 2**
**📝 Question:** Why did the group take inventory of the supplies?
✅ **Ground Truth Answer:** that the biggest earthquake had struck Los Angeles in the film This Is the End
💬 **Model Response:** Why did the group take inventory of the supplies? to ensure they had enough materials"

Answer: because

2. What did the teacher

----------------------------------------
🔹 **Sample 3**
**📝 Question:** What virus is causing millions of americans to be kept from work, school and public places?
✅ **Ground Truth Answer:** coronavirus
💬 **Model Response:** What virus is causing millions of americans to be kept from work, school and public places? coronavirus"

# R

In [20]:
import torch

def test_retain_set_responses(model, tokenizer, retain_set, num_samples=5, max_tokens=20):
    model.eval()
    device = "cuda"
    model.to(device)

    for i, sample in enumerate(retain_set[:num_samples]):
        question = sample["Question"]
        ground_truth = sample["Answer"]

        # Tokenize input and move to CUDA
        inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True).to(device)

        # Generate response with limited token output
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_tokens)
            response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Print formatted results
        print("\n----------------------------------------")
        print(f"🔹 **Sample {i+1}**")
        print(f"**📝 Question:** {question}")
        print(f"✅ **Ground Truth Answer:** {ground_truth}")
        print(f"💬 **Model Response (Max {max_tokens} Tokens):** {response}")

# Run the test on the retain set
test_retain_set_responses(reprojected_model, reprojected_tokenizer, retain_set, max_tokens=20)



----------------------------------------
🔹 **Sample 1**
**📝 Question:** Cnn news18 was among the news channels projecting what?
✅ **Ground Truth Answer:** this video
💬 **Model Response (Max 20 Tokens):** Cnn news18 was among the news channels projecting what? footage of the protest"

# Find the last time the news channel reported that the protest had

----------------------------------------
🔹 **Sample 2**
**📝 Question:** What is the name of the phantom thread film?
✅ **Ground Truth Answer:** Phantom Thread film
💬 **Model Response (Max 20 Tokens):** What is the name of the phantom thread film?
A: Phantom Thread film

What did some critics say about the film?
A:

----------------------------------------
🔹 **Sample 3**
**📝 Question:** What did muffy vestal develop?
✅ **Ground Truth Answer:** the lithium iodide cell
💬 **Model Response (Max 20 Tokens):** What did muffy vestal develop? a computer program that generates music"
    " by the gingore and grainchase"

--------------------------

# Eval

In [22]:
!pip install Rouge

In [23]:
from rouge import Rouge
import torch
from collections import defaultdict

def calculate_rouge_l(model, tokenizer, dataset, num_samples=50, max_tokens=20):
    """
    Computes the ROUGE-L score between model responses and ground truth answers for a given dataset.
    """
    model.eval()
    device = "cuda"
    model.to(device)

    rouge = Rouge()
    all_scores = []

    aspect_scores = defaultdict(list)  # Stores scores for each label type

    for i, sample in enumerate(dataset[:num_samples]):
        question = sample["Question"]
        ground_truth = sample["Answer"]
        label = sample["label"]  # Aspect label: Who, What, When, Why, Where

        # Tokenize input and move to CUDA
        inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True).to(device)

        # Generate response
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_tokens)
            response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Compute ROUGE-L score
        rouge_score = rouge.get_scores(response, ground_truth)[0]["rouge-l"]["f"]
        all_scores.append(rouge_score)
        aspect_scores[label].append(rouge_score)

    # Compute overall and aspect-wise average scores
    avg_rouge = sum(all_scores) / len(all_scores) if all_scores else 0.0
    avg_aspect_scores = {aspect: sum(scores) / len(scores) if scores else 0.0 for aspect, scores in aspect_scores.items()}

    return avg_rouge, avg_aspect_scores

# Evaluate Forget Set
forget_rouge, forget_aspect_scores = calculate_rouge_l(reprojected_model, reprojected_tokenizer, forget_set)

# Evaluate Retain Set
retain_rouge, retain_aspect_scores = calculate_rouge_l(reprojected_model, reprojected_tokenizer, retain_set)

# Display Results
print("\n========================================")
print("📊 **ROUGE-L Score Evaluation**")
print("========================================")
print(f"🛑 Forget Set ROUGE-L Score: {forget_rouge:.4f}")
print(f"✅ Retain Set ROUGE-L Score: {retain_rouge:.4f}")

print("\n🔎 **Aspect-Wise ROUGE-L Scores**")
print("----------------------------------------")
print("🔹 Forget Set Scores:")
for aspect, score in forget_aspect_scores.items():
    print(f"   - {aspect}: {score:.4f}")

print("\n🔹 Retain Set Scores:")
for aspect, score in retain_aspect_scores.items():
    print(f"   - {aspect}: {score:.4f}")

print("\n✅ If forgetting worked correctly, the Forget Set score should be **low**, while the Retain Set score should remain **high**.")



📊 **ROUGE-L Score Evaluation**
🛑 Forget Set ROUGE-L Score: 0.0726
✅ Retain Set ROUGE-L Score: 0.1086

🔎 **Aspect-Wise ROUGE-L Scores**
----------------------------------------
🔹 Forget Set Scores:
   - Who: 0.0364
   - Why: 0.0625
   - What: 0.0809
   - Where: 0.0823
   - When: 0.3018

🔹 Retain Set Scores:
   - What: 0.1087
   - Who: 0.0573
   - Where: 0.1393
   - When: 0.2113

✅ If forgetting worked correctly, the Forget Set score should be **low**, while the Retain Set score should remain **high**.
